In [1]:
from sqlalchemy import create_engine, String, inspect
import pandas as pd
from sqlalchemy.orm import DeclarativeBase, sessionmaker
from database import Base
from dotenv import load_dotenv
import os
import models


load_dotenv()
DATABASE_TEST_URL = os.getenv("DATABASE_TEST_URL")
engine = create_engine(DATABASE_TEST_URL)
Base.metadata.create_all(engine)

df = pd.read_csv("Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")



   




In [2]:
# Create an inspector object
inspector = inspect(engine)

# Get table names from the default 'public' schema
tables = inspector.get_table_names(schema="public")

print("Tables found:")
for table in tables:
    print(f"- {table}")

Tables found:
- dim_region
- fact_home_values


In [3]:
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
db = SessionLocal()
query = \
'''
SELECT * FROM dim_region
LIMIT 5;
'''
d = pd.read_sql(query, con = engine)
db.close()

In [4]:
print(d)

   region_id  size_rank      region_name region_type state_name
0     102001          0    United States     country        NaN
1     394913          1     New York, NY         msa         NY
2     753899          2  Los Angeles, CA         msa         CA
3     394463          3      Chicago, IL         msa         IL
4     394514          4       Dallas, TX         msa         TX


In [5]:
# See null counts for every column at once
df.isnull().sum()

RegionID      0
SizeRank      0
RegionName    0
RegionType    0
StateName     1
             ..
2026-01-31    0
2026-02-28    0
2026-03-31    0
2026-04-30    0
2026-05-31    0
Length: 322, dtype: int64

In [44]:
df_sizerank = df[df['RegionName'].isna()]
df_sizerank

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31


In [6]:
rename = {"RegionID": "region_id", 
          	"SizeRank": "size_rank",
            "RegionName": "region_name",
            "RegionType": "region_type",
            "StateName": "state_name"}

In [10]:
cols = ["region_id",	
            "size_rank",	
            "region_name", 
            "region_type",	
            "state_name"]
df = df.rename(rename, axis=1)
print(df.columns)
print(df.drop(cols, axis =1))

Index(['region_id', 'size_rank', 'region_name', 'region_type', 'state_name',
       '2000-01-31', '2000-02-29', '2000-03-31', '2000-04-30', '2000-05-31',
       ...
       '2025-08-31', '2025-09-30', '2025-10-31', '2025-11-30', '2025-12-31',
       '2026-01-31', '2026-02-28', '2026-03-31', '2026-04-30', '2026-05-31'],
      dtype='object', length=322)
        2000-01-31     2000-02-29     2000-03-31     2000-04-30  \
0    124226.850719  124445.405574  124716.445774  125297.773801   
1    220241.022643  221176.922725  222121.490928  224035.551610   
2    225669.811207  226509.589232  227628.165553  229853.756673   
3    156798.618056  156943.767900  157220.208333  157907.923431   
4    131332.841965  131391.338072  131458.585335  131633.219413   
..             ...            ...            ...            ...   
890            NaN            NaN            NaN            NaN   
891            NaN            NaN            NaN            NaN   
892  100581.808517  100839.003235  101317.8

In [20]:
from sqlalchemy import insert
with SessionLocal() as db:
        cols = ["region_id",	
                "size_rank",	
                "region_name", 
                "region_type",	
                "state_name"]
        df = df.rename(rename, axis=1)
        print(df.columns)
        records = df[cols].to_dict("records")

        # db.execute(insert(models.DimRegion), records)
        # db.commit()
        drop_cols = ["size_rank",	
                "region_name", 
                "region_type",	
                "state_name"]
        df_dropped = df.drop(drop_cols, axis=1)
        print(df_dropped.columns)
        id_col = ['region_id']
        date_cols= [c for c in df_dropped.columns if c not in id_col]

        long_df = df_dropped.melt(
                id_vars = id_col,
                value_vars=date_cols,
                var_name = "date",
                value_name = "value"
        )
        # print(long_df.head(5))
        long_df["date"] = pd.to_datetime(long_df["date"])
        long_df = long_df.dropna(subset=["value"])

        home_values = long_df.to_dict("records")
        db.execute(insert(models.FactHomeValues), home_values)
        db.commit()

Index(['region_id', 'size_rank', 'region_name', 'region_type', 'state_name',
       '2000-01-31', '2000-02-29', '2000-03-31', '2000-04-30', '2000-05-31',
       ...
       '2025-08-31', '2025-09-30', '2025-10-31', '2025-11-30', '2025-12-31',
       '2026-01-31', '2026-02-28', '2026-03-31', '2026-04-30', '2026-05-31'],
      dtype='object', length=322)
Index(['region_id', '2000-01-31', '2000-02-29', '2000-03-31', '2000-04-30',
       '2000-05-31', '2000-06-30', '2000-07-31', '2000-08-31', '2000-09-30',
       ...
       '2025-08-31', '2025-09-30', '2025-10-31', '2025-11-30', '2025-12-31',
       '2026-01-31', '2026-02-28', '2026-03-31', '2026-04-30', '2026-05-31'],
      dtype='object', length=318)
